## PropertyLens RAG — Notebook B: Inference (v4)

**Purpose:** query-time pipeline. Connects to a populated Pinecone index, runs the full V3 retrieval stack (hybrid + multi-query + weighted RRF + cross-encoder + MMR + reorder), and generates grounded answers with Gemma 3 via Ollama.

**Assumes:** Notebook A (`04_propertylens_build_index.ipynb`) has already run successfully. This notebook never touches raw CSVs, never builds chunks, never upserts. If the BM25 cache is missing, this notebook fails loudly — it does not try to fit one from scratch.

**Memory discipline:**
- Cross-encoder pinned to CPU (avoids MPS fighting with Gemma on Mac)
- Model bundle lazy-loaded (not imported until the first prediction query)
- Smoke test wrapped in try/except with RSS logged at each stage — if it ever crashes you'll see exactly which stage did it


### Install dependencies

In [1]:
# Lighter than Notebook A — no pyarrow needed, but joblib for the prediction model
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama joblib pandas numpy python-dotenv psutil

### Configuration

In [2]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ── Pinecone (must match Notebook A) ─────────────────────────────────────────
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY in repo-root .env"

# ── Models ────────────────────────────────────────────────────────────────────
DENSE_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"

# ── Retrieval knobs ───────────────────────────────────────────────────────────
TOP_K_RETRIEVAL = 50
TOP_K_RERANK    = 10
TOP_K_MMR       = 5
TOP_K_FINAL     = 5
MMR_LAMBDA      = 0.7
RRF_K           = 60
N_SUBQUERIES    = 3

# Source weights for weighted RRF — boosts amenity/xai chunks so they
# surface against the much-larger transactions pool.
SOURCE_WEIGHTS: dict[str, float] = {
    "transaction": 1.0,
    "amenity":     2.5,
    "trend":       1.0,
    "xai":         2.5,
}

# ── LLM ───────────────────────────────────────────────────────────────────────
OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"

# ── Device pinning ────────────────────────────────────────────────────────────
# Pin the cross-encoder to CPU. On a Mac running Ollama, MPS + Gemma + bi-encoder
# + cross-encoder compete for the same memory pool — forcing CE to CPU is the
# single most effective stability fix.
CROSS_ENCODER_DEVICE = "cpu"

# ── Paths (must match Notebook A) ────────────────────────────────────────────
def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root.")

REPO_ROOT         = find_repo_root()
BM25_CACHE_PATH   = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"
MODEL_BUNDLE_PATH = str(REPO_ROOT / "data" / "artifacts" / "hybrid_cluster_bundle.joblib")

print("Config loaded.")
print(f"  Pinecone index     : {PINECONE_INDEX}")
print(f"  BM25 cache path    : {BM25_CACHE_PATH}")
print(f"  Model bundle path  : {MODEL_BUNDLE_PATH}")
print(f"  CE device          : {CROSS_ENCODER_DEVICE}")
print(f"  Source weights     : {SOURCE_WEIGHTS}")


Config loaded.
  Pinecone index     : propertylens-rag
  BM25 cache path    : /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  Model bundle path  : /Users/bhuvesh/Documents/PropertyLens/data/artifacts/hybrid_cluster_bundle.joblib
  CE device          : cpu
  Source weights     : {'transaction': 1.0, 'amenity': 2.5, 'trend': 1.0, 'xai': 2.5}


### Memory helpers

Same `mem(label)` helper as Notebook A — RSS + `gc.collect()` for memory visibility.


In [3]:
from __future__ import annotations
import gc
import psutil

_PROC = psutil.Process(os.getpid())

def rss_mb() -> float:
    return _PROC.memory_info().rss / (1024 * 1024)

def mem(label: str) -> None:
    gc.collect()
    print(f"  [MEM] {label:<32s} RSS = {rss_mb():8.1f} MB")

mem("startup")


  [MEM] startup                          RSS =     72.7 MB


### Connect to Pinecone

No `create_index` — we expect Notebook A to have already populated it. If the index is empty, the retrieval stage will return no matches.


In [11]:
from __future__ import annotations
from pinecone import Pinecone

pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)

stats = index.describe_index_stats()
print(stats)
total = stats.get("total_vector_count") if isinstance(stats, dict) else getattr(stats, "total_vector_count", 0)
if not total:
    print("\n⚠ Pinecone index appears empty. Run Notebook A first.")


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '279',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 17 Apr 2026 05:13:52 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'amenities': {'vector_count': 85},
                'transactions': {'vector_count': 1995},
                'trends': {'vector_count': 312},
                'xai': {'vector_count': 204}},
 'storageFullness': 0.0,
 'total_vector_count': 2596,
 'vector_type': 'dense'}


### Load encoders

Dense encoder (BGE-M3) on default device. BM25 is loaded from cache — **if the cache file is missing this cell raises `FileNotFoundError`** rather than silently fitting a different encoder than the one used at upsert time.


In [4]:
from __future__ import annotations
import pickle
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder


def load_bm25_from_cache(cache_path: Path) -> BM25Encoder:
    if not cache_path.exists():
        raise FileNotFoundError(
            f"BM25 cache not found: {cache_path}\n"
            f"Run Notebook A (04_propertylens_build_index.ipynb) first to create it."
        )
    with cache_path.open("rb") as f:
        return pickle.load(f)


dense_encoder = SentenceTransformer(DENSE_MODEL_NAME)
bm25_encoder  = load_bm25_from_cache(BM25_CACHE_PATH)
print(f"Dense encoder : {DENSE_MODEL_NAME}")
print(f"BM25 encoder  : loaded from {BM25_CACHE_PATH}")
mem("after encoders loaded")


/Users/bhuvesh/Documents/PropertyLens/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36854.98it/s]


Dense encoder : BAAI/bge-m3
BM25 encoder  : loaded from /Users/bhuvesh/Documents/PropertyLens/notebooks/05_chatbot/bm25_encoder_v3.pkl
  [MEM] after encoders loaded            RSS =    987.6 MB


### Load cross-encoder (pinned to CPU)

Explicitly moving the cross-encoder to CPU sidesteps MPS/CUDA contention with Ollama. 10 passages × 512 tokens per query is well within CPU latency budgets.


In [5]:
from __future__ import annotations
from typing import Any, Tuple
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_cross_encoder(model_name: str, device: str) -> Tuple[Any, Any]:
    """Load cross-encoder tokenizer and model, pinned to device."""
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tok, model


ce_tokenizer, ce_model = load_cross_encoder(RERANKER_MODEL, CROSS_ENCODER_DEVICE)
print(f"Cross-encoder loaded on device: {CROSS_ENCODER_DEVICE}")
mem("after cross-encoder loaded")


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 16450.22it/s]

Cross-encoder loaded on device: cpu
  [MEM] after cross-encoder loaded       RSS =   1305.3 MB


### NLP filter extraction + namespace routing

Before retrieval, two lightweight pre-processing steps:

- **Filter extraction:** ask Gemma to extract `town`, `flat_type`, `sale_year` from the query, used as a Pinecone metadata filter on the transactions namespace only.
- **Namespace routing:** keyword-based selection of which namespaces to query. Always includes transactions; adds amenities/trends/xai based on keywords in the query.


In [6]:
from __future__ import annotations
import json, re
import ollama


def _set_ollama_host(base_url: str) -> None:
    os.environ["OLLAMA_HOST"] = base_url


def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """Use Gemma 3 to extract Pinecone metadata filters from a free-text query."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": ALL CAPS HDB town e.g. "TAMPINES", "BEDOK", "SERANGOON"
  - "flat_type": one of "2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"
  - "sale_year": integer year if mentioned
Omit any field you are not sure about. Return {{}} if nothing is clear.
Return ONLY JSON, no explanation.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if parsed else None
    except Exception as e:
        print(f"  Filter extraction failed: {e}")
        return None


def route_namespaces(query: str) -> list[str]:
    """Select Pinecone namespaces to query based on keywords."""
    q  = query.lower()
    ns = [NS_TRANSACTIONS]
    if any(kw in q for kw in ["mrt", "school", "mall", "hawker", "near", "amenity", "transport", "bus"]):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in ["trend", "rising", "falling", "increase", "decrease", "history",
                               "recent", "last year", "past", "over time", "appreciation"]):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in ["explain", "shap", "feature", "why", "reason", "driver",
                               "factor", "importan", "predict", "model say"]):
        ns.append(NS_XAI)
    return ns


# Smoke test
test_q = "Is $580k fair for a 4-room in Tampines?"
print(f"Query      : {test_q}")
print(f"Filters    : {extract_filters_from_query(test_q)}")
print(f"Namespaces : {route_namespaces(test_q)}")


Query      : Is $580k fair for a 4-room in Tampines?
Filters    : {'town': 'TAMPINES', 'flat_type': '4 ROOM'}
Namespaces : ['transactions']


### Hybrid retrieval + source-weighted RRF

Hybrid query = dense × alpha + sparse × (1−alpha). Two alpha extremes are run per query (dense-only and sparse-only), then fused via RRF.

`reciprocal_rank_fusion` uses per-source weights to prevent the massive transactions pool from drowning out amenity/xai chunks.


In [7]:
from __future__ import annotations
from typing import Any, Optional
import numpy as np


def _scale_sparse(sparse: dict, scale: float) -> dict:
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    index,
    query: str,
    alpha: float,
    top_k: int,
    namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict[str, Any]]:
    """Single Pinecone hybrid query. alpha=1.0 → pure dense, 0.0 → pure sparse."""
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * float(alpha)).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - float(alpha))
    res    = index.query(
        vector=dense, sparse_vector=sparse, top_k=int(top_k),
        namespace=namespace, include_metadata=True, filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id": getattr(m, "id", m.get("id")),
         "score": getattr(m, "score", m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


def retrieve_from_namespace(
    index,
    query: str,
    namespace: str,
    top_k: int,
    metadata_filter: Optional[dict] = None,
) -> tuple[list[dict], list[dict]]:
    """Run dense + sparse retrieval from one namespace. Returns (dense, sparse)."""
    dense  = _hybrid_query(index, query, alpha=1.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    sparse = _hybrid_query(index, query, alpha=0.0, top_k=top_k,
                           namespace=namespace, metadata_filter=metadata_filter)
    return dense, sparse


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    k: int = 60,
    source_weights: dict[str, float] | None = None,
) -> list[dict[str, Any]]:
    """
    Merge ranked lists using RRF with source-aware weights.

    score(d) = Σ  weight(source) × 1 / (k + rank_i(d))
    """
    weights = source_weights or SOURCE_WEIGHTS
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}
    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid = str(r.get("id", ""))
            if not rid:
                continue
            source = str((r.get("metadata") or {}).get("source", "transaction"))
            weight = weights.get(source, 1.0)
            scores[rid] = scores.get(rid, 0.0) + weight * (1.0 / (float(k) + float(rank)))
            if rid not in best:
                best[rid] = r
    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


print("Retrieval functions defined (source-weighted RRF active).")


Retrieval functions defined (source-weighted RRF active).


### Reranking funnel

Cross-encoder → MMR → lost-in-middle reorder. Each stage shrinks the candidate pool: 50 → 10 → 5.


In [8]:
from __future__ import annotations


def _get_text(candidate: dict, field: str = "parent_text") -> str:
    md = candidate.get("metadata") or {}
    return str(md.get(field) or md.get("parent_text") or "")


def rerank_cross_encoder(
    query: str,
    candidates: list[dict[str, Any]],
    tokenizer: Any,
    model: Any,
    top_k: int,
    device: str = CROSS_ENCODER_DEVICE,
) -> list[dict[str, Any]]:
    """Score (query, passage) pairs with the cross-encoder; return top_k."""
    if not candidates:
        return []
    pairs  = [(query, _get_text(c)) for c in candidates]
    inputs = tokenizer(pairs, padding=True, truncation=True,
                       max_length=512, return_tensors="pt")
    # Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


def _cosine(u: np.ndarray, v: np.ndarray) -> float:
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / denom) if denom > 1e-12 else 0.0


def mmr_filter(
    candidates: list[dict[str, Any]],
    query: str,
    top_k: int,
    lambda_param: float = 0.7,
) -> list[dict[str, Any]]:
    """Select top_k diverse candidates via Maximal Marginal Relevance."""
    if not candidates:
        return []
    texts   = [_get_text(c) for c in candidates]
    doc_emb = np.asarray(
        dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32
    )
    selected:  list[int] = []
    remaining: list[int] = list(range(len(candidates)))
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    selected.append(best0)
    remaining.remove(best0)
    while remaining and len(selected) < int(top_k):
        best_idx, best_val = None, -1e18
        for i in remaining:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(_cosine(doc_emb[i], doc_emb[j]) for j in selected)
            score   = lambda_param * rel - (1.0 - lambda_param) * max_sim
            if score > best_val:
                best_val, best_idx = score, i
        if best_idx is None:
            break
        selected.append(best_idx)
        remaining.remove(best_idx)
    return [candidates[i] for i in selected]


def reorder_for_context_window(
    candidates: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Lost-in-the-middle mitigation: best first, second-best last."""
    if len(candidates) <= 2:
        return list(candidates)
    ordered = sorted(
        candidates,
        key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)),
        reverse=True,
    )
    return [ordered[0], *ordered[2:], ordered[1]]


print("Reranking funnel defined.")


Reranking funnel defined.


### Multi-query retrieval

Generate N sub-queries via Gemma, then fan out: each (original + sub-query) × each routed namespace × (dense + sparse). All ranked lists fed into weighted RRF.


In [9]:
from __future__ import annotations


def generate_subqueries(
    query: str,
    n: int = N_SUBQUERIES,
    model: str = OLLAMA_MODEL,
) -> list[str]:
    """Generate n reformulations of the query using Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""You are a search query generator for Singapore HDB property data.
Generate {n} alternative search queries to help retrieve relevant data from a vector
database of HDB transactions, amenities, price trends, and SHAP features.
Return ONLY a numbered list. No explanations.

Original: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "")
        lines    = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception as e:
        print(f"  Sub-query generation failed: {e}")
        return []


def multi_query_retrieve(
    query: str,
    index,
    namespaces: list[str],
    top_k: int,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """Multi-query hybrid retrieval across all routed namespaces with weighted RRF."""
    all_queries = [query] + generate_subqueries(query)
    all_lists: list[list[dict]] = []
    for q in all_queries:
        for ns in namespaces:
            # Filter only applies to transactions (amenity/trend/xai don't have those metadata keys)
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            dense, sparse = retrieve_from_namespace(
                index, q, ns, top_k, filt,
            )
            all_lists.extend([dense, sparse])
    return reciprocal_rank_fusion(all_lists, k=RRF_K)


print("Multi-query retrieval defined.")


Multi-query retrieval defined.


### Full retrieval pipeline + defensive smoke test

The smoke test is wrapped in try/except with RSS logged at each stage. If anything crashes or hangs, the logs will tell you exactly which stage was running at the time.


In [12]:
from __future__ import annotations


def retrieve_and_rerank(
    query: str,
    index,
    verbose: bool = False,
) -> list[dict]:
    """Full RAG retrieval pipeline for a free-text query."""
    if verbose: mem("  retr: start")
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)
    if verbose: mem("  retr: after filter+route")

    fused = multi_query_retrieve(
        query=query, index=index, namespaces=namespaces,
        top_k=TOP_K_RETRIEVAL, metadata_filter=metadata_filter,
    )
    if verbose: mem(f"  retr: after fusion ({len(fused)} fused)")

    reranked = rerank_cross_encoder(
        query=query, candidates=fused[:TOP_K_RETRIEVAL],
        tokenizer=ce_tokenizer, model=ce_model, top_k=TOP_K_RERANK,
    )
    if verbose: mem(f"  retr: after CE rerank ({len(reranked)} ranked)")

    diverse = mmr_filter(
        candidates=reranked, query=query,
        top_k=TOP_K_MMR, lambda_param=MMR_LAMBDA,
    )
    if verbose: mem(f"  retr: after MMR ({len(diverse)} diverse)")

    final = reorder_for_context_window(diverse)[:TOP_K_FINAL]
    if verbose: mem("  retr: after reorder")
    return final


# ── Defensive smoke test ────────────────────────────────────────────────────
print("Smoke test (verbose mem tracking):")
try:
    smoke_ctx = retrieve_and_rerank(
        "Is $580k fair for a 4-room flat in Tampines?",
        index,
        verbose=True,
    )
    print(f"\n✓ Smoke test passed: {len(smoke_ctx)} chunks retrieved")
    for i, c in enumerate(smoke_ctx, 1):
        m = c.get("metadata") or {}
        print(f"  [{i}] {m.get('source')} | {m.get('town')} | "
              f"{m.get('flat_type','')} | {m.get('sale_year','')}")
except Exception as e:
    print(f"\n✗ Smoke test failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

mem("after smoke test")


Smoke test (verbose mem tracking):
  [MEM]   retr: start                    RSS =   1318.7 MB
  [MEM]   retr: after filter+route       RSS =   1318.7 MB
  [MEM]   retr: after fusion (59 fused)  RSS =   1405.0 MB
  [MEM]   retr: after CE rerank (10 ranked) RSS =   3189.0 MB
  [MEM]   retr: after MMR (5 diverse)    RSS =   3213.4 MB
  [MEM]   retr: after reorder            RSS =   3213.4 MB

✓ Smoke test passed: 5 chunks retrieved
  [1] transaction | TAMPINES | 4 ROOM | 2025
  [2] transaction | TAMPINES | 4 ROOM | 2022
  [3] transaction | TAMPINES | 4 ROOM | 2017
  [4] transaction | TAMPINES | 4 ROOM | 2021
  [5] transaction | TAMPINES | 4 ROOM | 2017
  [MEM] after smoke test                 RSS =   3213.4 MB


### Prediction tool (lazy-loaded)

The `HybridClusterEnsemble` model bundle is loaded only on first prediction request. If a session only asks retrieval-type questions (no prices to estimate), the model never consumes RAM.


In [13]:
from __future__ import annotations
import pandas as pd


# Lazy handle — actual joblib.load only runs when _get_bundle() is first called
_BUNDLE_CACHE: dict[str, object | None] = {"bundle": None, "attempted": False}


def _get_bundle():
    """Lazy-load the model bundle. Returns None if file is missing."""
    if _BUNDLE_CACHE["attempted"]:
        return _BUNDLE_CACHE["bundle"]
    _BUNDLE_CACHE["attempted"] = True
    if not os.path.exists(MODEL_BUNDLE_PATH):
        print(f"  Model bundle not found: {MODEL_BUNDLE_PATH}")
        return None
    import joblib
    bundle = joblib.load(MODEL_BUNDLE_PATH)
    _BUNDLE_CACHE["bundle"] = bundle
    print(f"  Model bundle loaded lazily: {type(bundle).__name__}")
    mem("after bundle load")
    return bundle


def extract_predict_request(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """Use Gemma to extract a PredictRequest-shaped dict from a free-text query."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract fields for a Singapore HDB price prediction.
Return ONLY a JSON object with these keys (omit if not mentioned):
  town               : ALL CAPS HDB town e.g. "TAMPINES"
  flat_type          : e.g. "4 ROOM", "3 ROOM"
  floor_area_sqm     : number (convert sqft to sqm if needed: sqft × 0.0929)
  storey_range       : string e.g. "07 TO 09"
  lease_commence_date: integer year lease started e.g. 1990
  flat_model         : e.g. "New Generation", "Improved", "Model A"
Return {{}} if nothing clear. Return ONLY JSON.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if (parsed.get("town") and parsed.get("flat_type")) else None
    except Exception as e:
        print(f"  PredictRequest extraction failed: {e}")
        return None


def run_prediction(predict_request: dict, bundle) -> str:
    """Call bundle.predict() and return a formatted result string."""
    try:
        row        = pd.DataFrame([predict_request])
        prediction = bundle.predict(row)
        price      = float(prediction[0]) if hasattr(prediction, "__len__") else float(prediction)
        low, high  = price * 0.90, price * 1.10
        return (
            f"Model price estimate: SGD {int(round(price)):,} "
            f"(confidence band: SGD {int(round(low)):,} – SGD {int(round(high)):,}). "
            f"Input used: {predict_request}"
        )
    except Exception as e:
        return f"[Prediction error: {type(e).__name__}: {e}]"


def prediction_tool(query: str) -> str:
    """Free-text query → price estimate string. Lazy-loads bundle on first call."""
    predict_request = extract_predict_request(query)
    if predict_request is None:
        return ""  # silently skip when query isn't predict-shaped
    bundle = _get_bundle()
    if bundle is None:
        return "[Prediction tool: model bundle unavailable.]"
    return run_prediction(predict_request, bundle)


print("Prediction tool defined (lazy-loaded on first use).")


Prediction tool defined (lazy-loaded on first use).


### Prompt builder + `generate_answer`

Same system prompt, context formatting, and Gemma call as V3. No behavioural change.


In [14]:
from __future__ import annotations


def build_system_prompt() -> str:
    """System prompt for Gemma 3 (same as V3)."""
    return """You are a Singapore HDB property pricing assistant for PropertyLens.
Help buyers and sellers make informed decisions about HDB resale prices.

Rules:
1. Answer ONLY using the provided context and model prediction (if given). No outside knowledge.
2. Cite every specific claim with [Context N] labels.
3. If a model prediction is provided, reference it explicitly in your answer.
4. If evidence is thin or contradictory, say so clearly.
5. Keep answers to 3-5 sentences unless detail is requested.
6. Give a Fair / Above market / Below market verdict ONLY for price fairness questions.
   For amenity, trend, or explanation questions, do NOT give a price verdict.
"""


def build_rag_prompt(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
) -> str:
    """Build the user-turn prompt with labelled context chunks and optional prediction."""
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md_  = c.get("metadata") or {}
        txt  = str(md_.get("parent_text") or "").strip()
        hdr  = f"[Context {i}] source={md_.get('source')} town={md_.get('town')} year={md_.get('sale_year')}"
        parts.extend([hdr, txt, ""])
    if prediction_result:
        parts.extend(["## Model prediction", prediction_result, ""])
    parts.extend(["## Question", query])
    return "\n".join(parts).strip()


def generate_answer(
    query: str,
    context_chunks: list[dict[str, Any]],
    prediction_result: str = "",
    model: str = OLLAMA_MODEL,
) -> str:
    """Grounded RAG answer using Ollama + Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": build_system_prompt()},
                {"role": "user",   "content": build_rag_prompt(query, context_chunks, prediction_result)},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {type(e).__name__}: {e}"


print("Prompt builder and generate_answer defined.")


Prompt builder and generate_answer defined.


### End-to-end demo

Same 8 queries V3 used — buyer, seller, trends, amenities, XAI, PropertyGuru listing, cross-source comparison, negotiation. Each demo call is independently wrapped so one failing demo doesn't skip the rest.


In [2]:
from __future__ import annotations

DEMO_QUERIES = [
    {"query": "Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?",
     "persona": "Buyer"},
    {"query": "What should I list my 5-room Bishan flat for given current market trends?",
     "persona": "Seller"},
    {"query": "Are HDB prices in Queenstown rising or falling over the last 3 years?",
     "persona": "Trends"},
    {"query": "What amenities are near Bedok North? Any MRT stations or schools?",
     "persona": "Amenities"},
    {"query": "Why did the model predict a high price for this Queenstown flat? What features drove it?",
     "persona": "XAI"},
    {"query": "Is $430,000 fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon? "
              "64 sqm, 52 years lease remaining, lease started 1978, "
              "3 mins walk to Serangoon MRT.",
     "persona": "PropertyGuru listing"},
    {"query": "Should I buy a 4-room flat in Tampines or Bedok? "
              "Compare prices, trends, and nearby amenities.",
     "persona": "Cross-source comparison"},
    {"query": "The seller is asking $650k for a 5-room in Sengkang. "
              "What is a reasonable counter-offer based on recent sales?",
     "persona": "Negotiation"},
]


def _print_chunk(i: int, c: dict) -> None:
    m      = c.get("metadata") or {}
    source = m.get("source", "")
    if source == "transaction":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] transaction | {m.get('town')} | {m.get('flat_type')} | {rp_str} | {m.get('sale_year')}")
    elif source == "amenity":
        print(f"    [{i}] amenity | {m.get('town')} | {m.get('amenity_type')} | count={m.get('count')}")
    elif source == "trend":
        rp     = m.get("resale_price")
        rp_str = f"${int(rp):,}" if isinstance(rp, (int, float)) else "N/A"
        print(f"    [{i}] trend | {m.get('town')} | median={rp_str} | {m.get('sale_year')}")
    elif source == "xai":
        preview = str(m.get("parent_text", ""))[:80]
        print(f"    [{i}] xai | type={m.get('xai_type')} | {preview}...")
    else:
        print(f"    [{i}] {source} | {m}")


def run_demo(demo: dict) -> None:
    """Run one demo query end-to-end. Wrapped so failures are isolated."""
    query = demo["query"]
    print(f"\n{'='*60}")
    print(f"[{demo['persona']}]")
    print(f"{query}")
    print(f"{'='*60}")
    try:
      
        metadata_filter = extract_filters_from_query(query)
        namespaces      = route_namespaces(query)
        print(f"  Filter     : {metadata_filter}")
        print(f"  Namespaces : {namespaces}")

        mem("before retrieve")
        ctx = retrieve_and_rerank(query, index)
        mem("after retrieve")
        print(f"\n  Context chunks ({len(ctx)}):")
        for i, c in enumerate(ctx, 1):
            _print_chunk(i, c)

        pred = prediction_tool(query)
        if pred:
            print(f"\n  Prediction : {pred[:140]}{'...' if len(pred) > 140 else ''}")

        answer = generate_answer(query, ctx, pred)
        print(f"\n  Answer:\n{answer}")
    except Exception as e:
        print(f"\n  ✗ Demo failed: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()


for demo in DEMO_QUERIES:
    run_demo(demo)

mem("after demos")



[Buyer]
Is $580k fair for a 4-room HDB in Tampines Street 81, Block 432?

  ✗ Demo failed: NameError: name 'extract_filters_from_query' is not defined

[Seller]
What should I list my 5-room Bishan flat for given current market trends?

  ✗ Demo failed: NameError: name 'extract_filters_from_query' is not defined

[Trends]
Are HDB prices in Queenstown rising or falling over the last 3 years?

  ✗ Demo failed: NameError: name 'extract_filters_from_query' is not defined

[Amenities]
What amenities are near Bedok North? Any MRT stations or schools?

  ✗ Demo failed: NameError: name 'extract_filters_from_query' is not defined

[XAI]
Why did the model predict a high price for this Queenstown flat? What features drove it?

  ✗ Demo failed: NameError: name 'extract_filters_from_query' is not defined

[PropertyGuru listing]
Is $430,000 fair for a 3-room HDB at 1 Lorong Lew Lian Serangoon? 64 sqm, 52 years lease remaining, lease started 1978, 3 mins walk to Serangoon MRT.

  ✗ Demo failed: NameE

Traceback (most recent call last):
  File "/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_77657/21498998.py", line 56, in run_demo
    metadata_filter = extract_filters_from_query(query)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'extract_filters_from_query' is not defined
Traceback (most recent call last):
  File "/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_77657/21498998.py", line 56, in run_demo
    metadata_filter = extract_filters_from_query(query)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'extract_filters_from_query' is not defined
Traceback (most recent call last):
  File "/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_77657/21498998.py", line 56, in run_demo
    metadata_filter = extract_filters_from_query(query)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'extract_filters_from_query' is not defined
Traceback (most recent call last):
  File "/var/folders/t4/jqyfdvcn0xn_qjk47

NameError: name 'mem' is not defined

### Notes

- **Restarting the kernel** is cheap now — no CSV loads, no chunk building. Re-running all cells takes ~30 seconds (BGE-M3 + cross-encoder downloads cached after first run).
- **If the smoke test memory report shows RSS climbing past ~8 GB on a 16 GB Mac**, your main culprit is usually Ollama holding Gemma in memory alongside the bi-encoder + cross-encoder. Try `ollama stop gemma3` between sessions or reduce `TOP_K_RERANK`.
- **To test with a fresh query:** just edit the smoke test cell's query string and re-run that cell. No re-initialisation needed.
- **Dependency on Notebook A:** this notebook will fail at the "load encoders" cell if `bm25_encoder_v3.pkl` is missing. That's intentional — silent re-fitting would produce a BM25 encoder that doesn't match the vocabulary used at upsert time.
